# Project 3 — SQLite Task Manager and Installable CLI

Build a small task-management application using SQLite, parameterized SQL, transactions, an argparse command-line interface, and unittest. A complete standalone Python package is included beside this notebook in `task_manager_package/`.

**Study time:** 3–5 hours · **Prerequisites:** Intermediate lessons 02, 14, 15, and 17.

## Requirements

- Create a task schema with id, title, and a validated completion flag.
- Add, list, complete, and remove tasks safely.
- Handle missing ids and empty titles with useful errors.
- Parse commands using argparse without consuming Jupyter's sys.argv.
- Build automated tests with independent in-memory databases.
- Extend the supplied src-layout package and document its CLI.

Run the cells in sequence. They use an in-memory database and cannot overwrite a real tasks file.

The runnable implementation is a worked example. Submit your own changes and regression tests in the independent section.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "projects"))
    sys.path.insert(0, os.getcwd())

## 1. Database schema and constraints

SQLite ships with Python. An in-memory connection is useful in tests because its data disappears on close. Database constraints protect stored state even when a caller forgets to validate input. AUTOINCREMENT supplies unique ids; CHECK ensures done is 0 or 1.

In [ ]:
import sqlite3

def initialize(connection):
    connection.execute("""
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL CHECK(length(trim(title)) > 0),
            done INTEGER NOT NULL DEFAULT 0 CHECK(done IN (0, 1))
        )
    """)
    connection.commit()

db = sqlite3.connect(":memory:")
initialize(db)
print(db.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='tasks'").fetchone())

**Design question:** Why do you need both application validation and database constraints?

## 2. Core database operations

Use placeholders, not string interpolation, to pass untrusted values to SQL. Domain functions should return useful values, such as a new task's id. Every update or deletion checks rowcount so a missing id cannot be silently ignored.

In [ ]:
def add_task(connection, title):
    if not isinstance(title, str) or not title.strip():
        raise ValueError("title cannot be empty")
    cursor = connection.execute("INSERT INTO tasks (title) VALUES (?)", (title.strip(),))
    connection.commit()
    return cursor.lastrowid

def list_tasks(connection, completed=None):
    query = "SELECT id, title, done FROM tasks"
    if completed is None:
        return connection.execute(query + " ORDER BY id").fetchall()
    return connection.execute(query + " WHERE done = ? ORDER BY id", (int(completed),)).fetchall()

def complete_task(connection, task_id):
    cursor = connection.execute("UPDATE tasks SET done = 1 WHERE id = ?", (task_id,))
    connection.commit()
    if cursor.rowcount == 0:
        raise KeyError("task not found")

def remove_task(connection, task_id):
    cursor = connection.execute("DELETE FROM tasks WHERE id = ?", (task_id,))
    connection.commit()
    if cursor.rowcount == 0:
        raise KeyError("task not found")

first = add_task(db, "Complete Intermediate lessons")
second = add_task(db, "Write tests")
complete_task(db, first)
print(list_tasks(db))

**Design question:** What does rowcount mean after UPDATE, and why check it?

## 3. Error paths and transaction boundaries

A transaction should commit a complete unit of work or roll it back on failure. For this simple tool each operation is small; larger workflows should coordinate multiple related updates in one transaction. Catch only expected errors.

In [ ]:
try:
    add_task(db, "   ")
except ValueError as error:
    print("Validation caught:", error)

remove_task(db, second)
print("Remaining:", list_tasks(db))

try:
    remove_task(db, 99999)
except KeyError as error:
    print("Missing id caught:", error)

**Design question:** How should you group two SQL changes that must succeed or fail together?

## 4. Command-line parsing without notebook side effects

argparse defines subcommands and values. In a notebook, pass a simulated command array; a real script reads sys.argv in its entry point. Keep the parser thin so domain behavior can be unit-tested separately.

In [ ]:
import argparse

def run_cli(argv, connection):
    parser = argparse.ArgumentParser(description="Tasks")
    commands = parser.add_subparsers(dest="command", required=True)
    add_parser = commands.add_parser("add")
    add_parser.add_argument("title")
    list_parser = commands.add_parser("list")
    list_parser.add_argument("--completed", action="store_true")
    done_parser = commands.add_parser("done")
    done_parser.add_argument("task_id", type=int)
    remove_parser = commands.add_parser("remove")
    remove_parser.add_argument("task_id", type=int)
    args = parser.parse_args(argv)

    if args.command == "add":
        return add_task(connection, args.title)
    if args.command == "list":
        return list_tasks(connection, completed=True if args.completed else None)
    if args.command == "done":
        complete_task(connection, args.task_id)
        return "completed"
    if args.command == "remove":
        remove_task(connection, args.task_id)
        return "removed"

print("Added id:", run_cli(["add", "Read the package README"], db))
print(run_cli(["list"], db))

**Design question:** Why is parsing command-line arguments separately from SQLite operations useful in tests?

## 5. Independent unit tests and cleanup

Test methods should not share an existing database connection or rely on previously executed notebook cells for their records. Each test initializes and closes its own in-memory database.

In [ ]:
import unittest

class TaskTests(unittest.TestCase):
    def setUp(self):
        self.connection = sqlite3.connect(":memory:")
        initialize(self.connection)

    def tearDown(self):
        self.connection.close()

    def test_add_complete(self):
        task_id = add_task(self.connection, "Learn")
        complete_task(self.connection, task_id)
        self.assertEqual(list_tasks(self.connection, completed=True)[0][1], "Learn")

    def test_blank_title(self):
        with self.assertRaises(ValueError):
            add_task(self.connection, " ")

    def test_missing_id(self):
        with self.assertRaises(KeyError):
            complete_task(self.connection, 987654)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TaskTests)
assert unittest.TextTestRunner(verbosity=0).run(suite).wasSuccessful()
db.close()

**Design question:** Why is an isolated in-memory database preferable for unit tests?

## 6. Companion installable package

The folder [task_manager_package](task_manager_package/README.md) contains a real Python package using a src layout, pyproject.toml, a task_manager.cli entry point, and unittest tests. It writes to a user-selected SQLite database file when the CLI is run, so choose a safe test path.

From the package directory:

```bash
python -m pip install -e .
intermediate-tasks --database test_tasks.db add "Review SQLite"
intermediate-tasks --database test_tasks.db list
python -m unittest discover -s tests -v
```

This notebook avoids persistent files; the CLI intentionally uses a database file to demonstrate persistence.

## Independent project extensions

**Extension 1:** Add priority and due date columns with suitable validation and an upgrade/migration strategy.

**Review evidence.** For the stats command, show the output of an empty database and one containing one open and two completed tasks. Check the counts after a task changes status. Explain the decision behind your boundary case.

In [ ]:
# TODO: implement extension 1 and its tests.

**Extension 2:** Add a stats command that summarizes completed and incomplete tasks.

Expose stats through the installable CLI and its query layer. Return open and done counts, including zeros. Add a subprocess regression against a temporary database, then reopen it and verify the stats command made no writes. Explain which module owns the connection.

In [ ]:
# TODO: implement extension 2 and its tests.

**Extension 3:** Export and import tasks as JSON, rejecting malformed input and duplicates.

In [ ]:
# TODO: implement extension 3 and its tests.

**Extension 4:** Write integration tests for the CLI in addition to pure core tests.

In [ ]:
# TODO: implement extension 4 and its tests.

## Review checklist

- [ ] Task operations use SQL placeholders.
- [ ] Title validation and missing-id cases have tests.
- [ ] No test modifies a real user database.
- [ ] CLI parsing can be tested with an explicit argument list.
- [ ] Resources are closed; transactions are deliberate.
- [ ] The package builds and the CLI's README is clear.